In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

In [ ]:
SRC=Path.cwd().absolute(); TASK=SRC.parent; ROOT=TASK.parent
INPUT=TASK/'input'; OUTPUT=TASK/'output'
CFG=json.loads((ROOT/'config.json').read_text()); SEED=CFG['seed']
np.random.seed(SEED); torch.manual_seed(SEED)
sys.path.insert(0, str(ROOT/'c_fit'/'src'))

In [ ]:
from score_models import evaluate, contrasts

In [ ]:
arrays=np.load(INPUT/'prepare'/'windows.npz')
selected=json.loads((INPUT/'fit'/'selection.json').read_text())
one_meta=pd.read_csv(INPUT/'prepare'/'one_test_meta.csv')
path_meta=pd.read_csv(INPUT/'prepare'/'path_test_meta.csv')

In [ ]:
one, path=evaluate(CFG, arrays, (one_meta,path_meta), selected,
                   INPUT/'fit', INPUT/'prepare', OUTPUT)

In [ ]:
paired=contrasts(CFG, one, path)
paired.to_csv(OUTPUT/'paired_contrasts.csv', index=False)
display(one.groupby('config')[['nll','crps','mae_log','cover90','tail_brier']].mean().sort_values('nll'))
display(paired[['contender','baseline','metric','delta','low95','high95','plots']])